# Two-Scales Volatility and the Noise

The two-scales estimator of [Zhang, Mykland and Aït-Sahalia (2005)](https://doi.org/10.1198/016214505000000169) uses every tick and still removes the bias of microstructure noise. This tutorial computes it with `two_scales_variance` of TwoScalesVolatility.jl, rebuilds it by hand from its formula, chooses the number of sub-grids by the rule of the article, and raises the noise to see where sparse sampling fails and two scales hold.

In [ ]:
Pollis.packages("TwoScalesVolatility" => "https://github.com/Trumpingtons/TwoScalesVolatility.jl", "Plots")

## The Same Simulated Day

A trading day with a tick every second, a daily volatility of 1% and noise of 2 basis points, as in the first tutorial.

In [ ]:
using TwoScalesVolatility, Random, Statistics
import Plots

rng = Xoshiro(7)
n = 23400  # a tick every second over a trading day of 6.5 hours
efficient = log(100) .+ cumsum(0.01 / sqrt(n) .* randn(rng, n))  # efficient log price, daily volatility 1%
noise = 0.0002 .* randn(rng, n)  # microstructure noise of 2 basis points
y = efficient .+ noise  # the observed log prices, one a second
nret = length(y) - 1  # tick returns

## Two-Scales Volatility in One Call

`two_scales_variance(y, K)` gives the two-scales estimate of the day's integrated variance from the log prices `y` with `K` sub-grids, adjusted for small samples: equations (55) and (64) of the article. Its square root is the daily volatility. The article chooses K = c* n^(2/3), with c* from its equation (10), which needs the noise variance and the quarticity, the integral of the fourth power of the volatility. Taking the day as T = 1 and its volatility as nearly constant, the quarticity is close to the integrated variance squared, which five-minute returns estimate first: `sparse_variance(y, 300)` takes every 300th price.

In [ ]:
first_guess = sparse_variance(y, 300)  # integrated variance from five-minute returns, a first estimate
K = clamp(round(Int, optimal_c(1.0, noise_variance(y), first_guess^2) * nret^(2 / 3)), 2, nret ÷ 2)  # K = c* n^(2/3), equation (10)
(two_scales = sqrt(two_scales_variance(y, K)), K = K, noise_basis_points = 1e4 * sqrt(noise_variance(y, K)), true_volatility = 0.01, true_noise_basis_points = 2.0)

## The Estimator by Hand

The slow scale averages realized variance over K offset sub-grids, each using every K-th tick, so no tick is discarded. The fast scale is realized variance on every tick, which is mostly noise. With n tick returns and n_bar = (n − K + 1) / K returns per sub-grid, the estimator is TSRV = RV_avg − (n_bar / n) × RV_all, equation (55), and equation (64) divides it by 1 − n_bar / n for small samples. RV_all / (2n) estimates the noise variance, equation (22).

In [ ]:
function tsrv(p, K)  # the two-scales realized variance of the article, by hand
	nret = length(p) - 1  # tick returns
	RV_all = sum(diff(p) .^ 2)  # fast scale: every tick
	RV_avg = mean(sum(diff(p[k:K:end]) .^ 2) for k in 1:K)  # slow scale: average over K sub-grids
	n_bar = (nret - K + 1) / K
	return (TSRV = (RV_avg - (n_bar / nret) * RV_all) / (1 - n_bar / nret), RV_avg = RV_avg, noise = RV_all / (2 * nret))
end
hand = tsrv(y, K)
(by_hand = sqrt(hand.TSRV), package = sqrt(two_scales_variance(y, K)), slow_scale_alone = sqrt(hand.RV_avg), noise_by_hand_basis_points = 1e4 * sqrt(hand.noise))

## How the Number of Sub-grids Matters

K sets the slow time scale: with ticks one second apart, K = 300 makes each sub-grid return a five-minute return. Too few sub-grids leave a large correction to make; too many leave few returns in each sub-grid. The dotted line marks the K of the article's rule.

In [ ]:
Ks = [3, 10, 30, 100, 300, 600, 1500]
estimates = [sqrt(two_scales_variance(y, k)) for k in Ks]
Plots.plot(Ks, 100 .* estimates; xscale = :log10, marker = :circle, xlabel = "sub-grids K", ylabel = "volatility (%)", label = "two-scales", title = "Two-scales volatility against K")
Plots.hline!([1.0]; linestyle = :dash, label = "true volatility")
Plots.vline!([K]; linestyle = :dot, label = "K = c* n^(2/3)")

## More Noise

A day like the one above is now simulated 100 times at each noise level, K following the rule of the article each day. Realized volatility at five minutes rises with the noise, as each five-minute return still holds the noise of two prices. The two-scales estimate stays close to the true volatility of 1% on average, while its spread across days grows with the noise: the first guess of the quarticity comes from five-minute returns, which the noise inflates, so K comes out smaller than the true values would give.

In [ ]:
function day(seed, sd)  # a simulated day of log prices, volatility 1% and noise sd
	rng = Xoshiro(seed)
	return log(100) .+ cumsum(0.01 / sqrt(n) .* randn(rng, n)) .+ sd .* randn(rng, n)
end
rows = map([0.0, 0.0002, 0.0005, 0.001, 0.002]) do sd
	days = [day(seed, sd) for seed in 1:100]
	Ks = [clamp(round(Int, optimal_c(1.0, noise_variance(p), sparse_variance(p, 300)^2) * nret^(2 / 3)), 2, nret ÷ 2) for p in days]
	five = [sqrt(sparse_variance(p, 300)) for p in days]
	two = [sqrt(max(two_scales_variance(p, k), 0.0)) for (p, k) in zip(days, Ks)]
	(noise_basis_points = 1e4 * sd, median_K = median(Ks), realized_five_minutes = round(mean(five), sigdigits = 3), two_scales = round(mean(two), sigdigits = 3), two_scales_spread = round(std(two), sigdigits = 2))
end

## Things to Try

- Make the noise correlated across ticks, for instance a moving average of the draws: the noise estimate and the correction are then only approximate.
- Thin the day to one tick every ten seconds and see how few ticks two scales needs.
- Run **Microstructure Noise** in High-Frequency Volatility on your own ticks, to see how much of one tick return is noise.